# 01 - EDA Current Stress
EDA + tracking MLflow + output lokal.

In [17]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import importlib.util

import mlflow
import matplotlib.pyplot as plt

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
UTILS_PATH = next((d / "mlflow_utils.py" for d in CANDIDATE_DIRS if (d / "mlflow_utils.py").exists()), None)
if UTILS_PATH is None:
    raise FileNotFoundError("mlflow_utils.py tidak ditemukan. Pastikan notebook dijalankan dari root repo atau folder experiments.")

spec = importlib.util.spec_from_file_location("cs_mlflow_utils", UTILS_PATH)
cs_utils = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(cs_utils)

RANDOM_STATE = cs_utils.RANDOM_STATE
configure_mlflow = cs_utils.configure_mlflow
set_seeds = cs_utils.set_seeds
load_current_stress_dataset = cs_utils.load_current_stress_dataset
get_dataset_path = cs_utils.get_dataset_path
log_run_metadata = cs_utils.log_run_metadata
create_local_output_dir = cs_utils.create_local_output_dir

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)


MlflowException: Cannot set a deleted experiment 'Current Stress' as the active experiment. You can restore the experiment, or permanently delete the experiment to create a new one.

In [ ]:
NOTEBOOK_NAME = "01_eda_current_stress.ipynb"
raw_df, feature_df, y = load_current_stress_dataset(repo_root)
dataset_source = str(get_dataset_path(repo_root))

with mlflow.start_run(run_name="EDA - Current Stress") as run:
    log_run_metadata(
        run_description="EDA current stress; dataset=current_stress_v1; split metadata=80/20",
        tags={"features": "all", "model": "EDA"},
        params={"stage": "eda", "rows": len(raw_df), "columns": raw_df.shape[1]},
        dataset_df=raw_df.assign(target=y.values),
        dataset_context="eda",
        dataset_source=dataset_source,
    )

    local_output_dir = create_local_output_dir(repo_root, NOTEBOOK_NAME, run.info.run_id)
    raw_df.describe(include="all").transpose().to_csv(local_output_dir / "eda_describe.csv")
    raw_df.isna().sum().rename("missing_count").to_csv(local_output_dir / "missing_values.csv")

    fig, ax = plt.subplots(figsize=(6, 4))
    raw_df["Stress_Level"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title("Stress Label Distribution")
    fig.tight_layout()
    fig.savefig(local_output_dir / "stress_label_distribution.png", dpi=150)
    plt.close(fig)

    mlflow.log_artifacts(str(local_output_dir), artifact_path="local_outputs")
    print("Run ID:", run.info.run_id)
    print("Local output:", local_output_dir)
